- PDF 랜더링 :PyMuPDF(fitz) 사용해서 pdf페이지를 PNG 이미지로 변환 및 저장
- 비전 기반 페이지 요약 : gpt-5.4-nano 각 페이지를 해석해서 표 구조를 포함한 텍스트 요약문을 도출
- 요약-이미지 인덱싱 : 요약문 텍스트는 chromadb 임베딩저장 메타데이터에 원본 이미지 경로를 매핑
- 검색 및 리트리버 : 질문에 맞는 페이지 요약문이 탐지되면 관련 원본 페이지 이미지를 확보
- 멀티모달 답변 생성 : gpt-4.5-nano 질문과 원본페이지 이미지를 동시에 llm에 제공해서 시각적인 데이터(표구조포함)를 직접 해석한 답변을 완성
- 웹검색 Fallback연동 : 내부지식이 부족한 경우 네이버openai/duckduckgo 실시간 웹 검색으로 답변을 보강

### STEP1: 환경설정

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
os.getenv('OPENAI_API_KEY')[:20]

'sk-proj-2y5DJ_EowWqi'

### STEP2: PyMuPDF(fitz)를 이용한 PDF의 고화질 이미지 변환

In [2]:
import fitz
import glob
os.makedirs('./docs_images', exist_ok=True)
pdf_files = glob.glob("./doc/*.pdf")
print('pdf 목록: ', pdf_files)
page_images = []

for pdf in pdf_files:
    print(f'이미지 렌더링 중...{pdf}')
    doc = fitz.open(pdf)
    base_name = os.path.splitext(os.path.basename(pdf))[0]
    
    for page_idx in range(len(doc)):
        page = doc.load_page(page_idx)
        pix = page.get_pixmap(dpi=150)
        image_path = f'./docs_images/page_{base_name}_{page_idx}.png'
        pix.save(image_path)
        page_images.append({
            "image_path":image_path,
            "source":pdf,
            "page":page_idx
        })

print(f"총 {len(page_images)}개의 pdf 페이지 이미지 렌더링 완료")

pdf 목록:  ['./doc\\01._북한_원산갈마해안관광특별구법에_대한_법적_고찰.pdf', './doc\\02._인도네시아_헌법의_특성과_체계.pdf', './doc\\03._미등록_아동_방지를_위한_출생등록될_권리의_법제화_방안.pdf', './doc\\법제시론_국회_입법기능의_정상화_전학선.pdf']
이미지 렌더링 중..../doc\01._북한_원산갈마해안관광특별구법에_대한_법적_고찰.pdf
이미지 렌더링 중..../doc\02._인도네시아_헌법의_특성과_체계.pdf
이미지 렌더링 중..../doc\03._미등록_아동_방지를_위한_출생등록될_권리의_법제화_방안.pdf
이미지 렌더링 중..../doc\법제시론_국회_입법기능의_정상화_전학선.pdf
총 139개의 pdf 페이지 이미지 렌더링 완료


### STEP3: GPT-5.4-NANO 비전 모델을 이용한 페이지별 구조화 요약 및 Chroma 인덱싱

In [3]:
import base64
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import shutil

def encode_image(image_path):
    with open(image_path, 'rb') as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')
print(f'gpt-5.4-nano 비전을 통한 pdf 페이지 해석 및 요약 생성 시작...')    
llm_model = 'gpt-5.4-nano'
llm_vision = ChatOpenAI(model=llm_model,temperature=0,max_completion_tokens=1024)
summarized_docs = []
for item in page_images[:10]:  # 테스트를 위해서 10개 pdf 이미지만 처리
    image_path = item['image_path']
    base64_image = encode_image(image_path)
    
    prompt_message = HumanMessage(
        content=[
            {'type':'text', 
             'text':'pdf 페이지의 이미지를 상세히 요약해 주세요, 페이지 내에 표(Table)나 그래프, 다이그램이 존재한다면'
             ', 그 안의 데이터를 하나도 누락하지 말고 마크다운 테이블 형태(|컬럼1|컬럼2|)로 상세하게 표기해주세요.'
             '시각적인 정보와 텍스트 문맥을 분석하여 정보가 손실되지 않도록 작성해 주세요, 한국어로 작성해 주세요'},
            {'type':'image_url',
             'image_url':{
                 'url': f"data:image/png;base64,{base64_image}"
             }}
        ]
    )
    response = llm_vision.invoke([prompt_message])
    summary_text = response.content
    doc = Document(
        page_content=summary_text,
        metadata = {
            'image_path':image_path,
            'source':item['source'],
            'page':item['page']
        }
    )
    summarized_docs.append(doc)
    print(f' - 요약완료 {image_path} 크기:{len(summary_text)}자')
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')    
persist_dir_vision = './chroma_db_vision'

if os.path.exists(persist_dir_vision):
    shutil.rmtree(persist_dir_vision) 
    
vectorstore_vision = Chroma.from_documents(
    documents=summarized_docs,
    embedding=embeddings,
    persist_directory=persist_dir_vision,
    collection_name='vision_db'
)
retriever_vision = vectorstore_vision.as_retriever(search_kwargs = {'k':2})
print('비전 RAG용 ChromaDB 구축 완료')

c:\Users\Playdata\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


gpt-5.4-nano 비전을 통한 pdf 페이지 해석 및 요약 생성 시작...
 - 요약완료 ./docs_images/page_01._북한_원산갈마해안관광특별구법에_대한_법적_고찰_0.png 크기:762자
 - 요약완료 ./docs_images/page_01._북한_원산갈마해안관광특별구법에_대한_법적_고찰_1.png 크기:396자
 - 요약완료 ./docs_images/page_01._북한_원산갈마해안관광특별구법에_대한_법적_고찰_2.png 크기:1712자
 - 요약완료 ./docs_images/page_01._북한_원산갈마해안관광특별구법에_대한_법적_고찰_3.png 크기:1738자
 - 요약완료 ./docs_images/page_01._북한_원산갈마해안관광특별구법에_대한_법적_고찰_4.png 크기:1701자
 - 요약완료 ./docs_images/page_01._북한_원산갈마해안관광특별구법에_대한_법적_고찰_5.png 크기:1700자
 - 요약완료 ./docs_images/page_01._북한_원산갈마해안관광특별구법에_대한_법적_고찰_6.png 크기:1732자
 - 요약완료 ./docs_images/page_01._북한_원산갈마해안관광특별구법에_대한_법적_고찰_7.png 크기:1632자
 - 요약완료 ./docs_images/page_01._북한_원산갈마해안관광특별구법에_대한_법적_고찰_8.png 크기:1688자
 - 요약완료 ./docs_images/page_01._북한_원산갈마해안관광특별구법에_대한_법적_고찰_9.png 크기:1698자
비전 RAG용 ChromaDB 구축 완료


### STEP4: LangGraph

In [4]:
from typing import List, Dict, Any
from typing_extensions import TypedDict
import urllib.request
import urllib.parse
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun

class GraphState(TypedDict):
    question:str
    generation:str
    # web_search:str
    web_search:bool
    documents:List[Document]

class GradeDocument(BaseModel):
    binary_score:str = Field(
        description="문서 요약이 질문과 실질적인 관련이 있는지 여부, 'yes' 또는 'no' "
    )

def retrieve(state:GraphState)->Dict[str,Any]:
    print(f'[node:retrieve] 유사 요약 페이지 검색 시작...')
    question = state['question']
    documents = retriever_vision.invoke(question)
    print(f'[Node: retrieve] 검색 완료 매핑된 이미지 페이지수: {len(documents)}')
    return {"documents":documents,'question':question}

def grade_documents(state:GraphState)->Dict[str,Any]:
    print(f'[node:grade_documents] 페이지 요약 적합성 판정...')
    question = state['question']
    documents = state['documents']
    llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0, max_completion_tokens=1024)
    structed_llm_grader = llm.with_structured_output(GradeDocument)
    
    system_prompt = '''당신은 사용자의 질문과 문서 요약의 의미론적 관련성을 채점하는 위원입니다.
    요약문의 질문에 대한 내용이나 표 구조의 힌트가 들어있다면 'yes' 로 채점하세요. 전혀 관계가 없다면 'no' 입니다.'''
    grade_prompt = ChatPromptTemplate.from_messages([
        ('system',system_prompt),
        ('human',"문서요약:\n\n{document}\n\n 사용자 질문:{question}")
    ])
    grader_chain = grade_prompt | structed_llm_grader
    filtered_documents = []
    web_search_flag = False

    for i, doc in enumerate(documents):
        res = grader_chain.invoke({'question':question, 'document':doc.page_content})
        score = res.binary_score
        if score == 'yes':
            print(f"[페이지 {doc.metadata.get('page')+1}] 심사 결과 : 적합")
            filtered_documents.append(doc)
        else:
            print(f"[페이지 {doc.metadata.get('page')+1}] 심사 결과 : 부적합")
    if not filtered_documents:
        print(' 검색문서가 부적합함으로 실시간 인터넷검색을 가동합니다.')
        web_search_flag=True
    else:
        print(f" {len(filtered_documents)}개의 유효한 페이지 비주얼 문맥을 확보했습니다.")
    return {"documents":filtered_documents,'question':question,"web_search":web_search_flag}

import json

def web_search(state:GraphState)->Dict[str,Any]:
    print("[node: web_search] 실시간 포털 검색 가동중...")
    question = state['question']
    documents = state.get('documents',[])
    search_results = ""
    used_source=""
    client_id = os.getenv('NAVER_CLIENT_ID')
    client_secret = os.getenv('NAVER_CLIENT_SECRET')
    if client_id and client_secret:
        print('naver api 뉴스 검색중....')
        try:
            encText = urllib.parse.quote(question)
            url = f"https://openapi.naver.com/v1/search/news.json?query={encText}&display=5"
            req = urllib.request.Request(url)
            req.add_header("X-Naver-Client-Id", client_id)
            req.add_header("X-Naver-Client-Secret", client_secret)
            response = urllib.request.urlopen(req)

            if response.getcode() == 200:
                res_body = response.read()
                data = json.loads(res_body.decode('utf-8'))
                items = data.get('items', [])
                snippets = []
                
                for item in items:
                    title = item.get('title', '').replace('<b>', '').replace('</b>', '')
                    description = item.get('description', '').replace('<b>', '').replace('</b>', '')
                    snippets.append(f"제목: {title}\n요약: {description}")
                
                if items:                                         
                    search_results = "\n\n".join(snippets)
                    used_source = "web_search_naver_news"
                    print(f"  - 네이버 실시간 뉴스 수집 완료! ({len(items)}건)")
                else:                                           
                    print("  - 네이버 뉴스 검색 결과 없음, DuckDuckGo로 전환")

        except Exception as e:
            print(f"  - 네이버 API 호출 중 예외: {e}")

    if not search_results:
        print('DuckDuckGo 백업 검색 가동중...')
        try:
            search = DuckDuckGoSearchRun()
            search_results = search.run(question)
            used_source='web_search_duckduckgo'
        except Exception as e:
            print('DuckDuckGo 오류', e)
    
    if not search_results:
        search_results = '실시간 웹 검색 서비스 (news, duckduckgo)의 최신 정보를 수집하지 못했습니다.'\
        '기존 내부 데이터 베이스를 바탕으로 답변을 제공합니다'
        used_source = 'callback_knowlege_base'

    web_doc = Document(
        page_content=search_results,
        metadata = {'source':used_source}
    )

    documents.append(web_doc)
    return {'documents':documents, 'question':question}

def generation(state:GraphState)->Dict[str,Any]:
    print("[node: generation] 비전 분석기 기반 답변 구성 중...")
    question = state['question']
    documents = state['documents']
    # llm_vision = ChatOpenAI(model='gpt-5.4-nano',temperature=0,max_completion_tokens=1024)
    image_contents, text_contents = [], []

    for doc in documents: 
        image_path = doc.metadata.get('image_path')
        if image_path and os.path.exists(image_path):
            base64_image = encode_image(image_path)
            image_contents.append({
                'type':'image_url',
                'image_url':{
                    'url':f"data:image/png;base64,{base64_image}"
                }
            })
            text_contents.append(f"[페이지: {doc.metadata.get('page')+1} 요약]: {doc.page_content}")
        else:
            text_contents.append(f"[출처: {doc.metadata.get('source','알수없음')}]: {doc.page_content}")
    system_prompt = '''당신은 고성능 Q&A 어시스턴트 입니다.
    제공된 이미지 파일들 (원본 pdf의 렌더링 스크린샷)을 직접 확인하고 질문에 대한 정확한 답을 이미지 안의 표, 선, 숫자 구조를 보고 도출해 주세요.
    이미지가 없고 텍스트 컨테스트만 주어진 경우 텍스트를 최우선으로 사용하여 상세하게 답변하세요.
    답변은 한국어로 일목요연하고 설득력 있게 구성해 주세요.
    
    텍스트 요약 컨텍스트:
    ''' + "\n\n".join(text_contents)

    human_content = [{"type":"text", "text":f"Question: {question}"}]
    human_content.extend(image_contents)

    messages = [
        ("system", system_prompt),
        HumanMessage(content=human_content)
    ]

    response = llm_vision.invoke(messages)
    print("멀티모달 최종 답변 구성 완료")
    
    return({"documents":documents, "question":question, "generation":response.content})    

C:\Users\Playdata\AppData\Local\Temp\ipykernel_17568\2262125133.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


### STEP5: 조건부 엣지 선언 및 워크플로우 구성

In [5]:
from langgraph.graph import StateGraph, END
def decide_to_generate(state:dict)->str:
    print("[Edge: Decide] 경로 분기 판단...")
    if state['web_search']:
        print("관련 비전 데이터 부재로 실시간 검색으로 우회....")
        return 'web_search'
    else:
        print("유혀 비전 페이지 발견으로 최종 생성으로 연결....")
        return 'generate'

workflow = StateGraph(GraphState)    
workflow.add_node('retrieve',retrieve)
workflow.add_node('grade_documents',grade_documents)
workflow.add_node('web_search',web_search)
workflow.add_node('generate',generation)

workflow.set_entry_point('retrieve')
workflow.add_edge('retrieve','grade_documents')
workflow.add_conditional_edges(
    'grade_documents',
    decide_to_generate,
    {
        'web_search':'web_search',
        'generate':'generate'
    }
)
workflow.add_edge('web_search','generate')
workflow.add_edge('generate',END)

app_vision = workflow.compile()
print('비젼 컴파일 완성')

비젼 컴파일 완성


### 테스트 시나리오 1: 내부 pdf 문서의 표 구조 해석 테스트

In [6]:
query = '원산갈마해안관광특별구법 배경과 제재 규정 구조 를 상세히 설명해 주세요'
inputs = {'question':query}
config = {'recursion_limit':50}
result = app_vision.invoke(inputs, config)
print("==================================== 최종 답변 ===============================")
print(result['generation'])
print("=============================================================================")

[node:retrieve] 유사 요약 페이지 검색 시작...
[Node: retrieve] 검색 완료 매핑된 이미지 페이지수: 2
[node:grade_documents] 페이지 요약 적합성 판정...
[페이지 5] 심사 결과 : 적합
[페이지 6] 심사 결과 : 적합
 2개의 유효한 페이지 비주얼 문맥을 확보했습니다.
[Edge: Decide] 경로 분기 판단...
유혀 비전 페이지 발견으로 최종 생성으로 연결....
[node: generation] 비전 분석기 기반 답변 구성 중...
멀티모달 최종 답변 구성 완료
==================================== 최종 답변 ===============================
아래 설명은 업로드된 이미지(해당 PDF의 본문 페이지 5~6)에서 **“원산갈마해안관광특별구법” 배경**과 그와 연결된 **대북 제재 체제(Sanctions Regime) 및 제재 관련 규정 구조(본문에서 ‘법적 쟁점/제재 영향’으로 서술된 부분)**를, 이미지에 나온 문장·근거 번호(각주 1~6 등) 중심으로 구조화해 정리한 것입니다.

---

## 1) 법(원산갈마해안관광특별구법)의 배경: “제재 체제 하의 관광지구 개발”이 핵심 문제의식
이미지(페이지 5)의 서론은 남북관계 맥락에서 다음 논리 흐름을 전개합니다.

1. **남북관계의 구조적 고착**  
2. 그 결과 북한의 대외 환경 속에서 **관광정책 및 대규모 관광지구 개발사업**이 주목됨  
3. 그런데 과거 금강산 등과 같은 “제한적 협력”과 달리, 현재 관광사업은 **단순 교류가 아니라 ‘법적·제도적 쟁점’**을 갖는다고 문제 제기  
4. 특히 그 쟁점의 핵심을 “**대북 제재 체제(Sanctions Regime)의 장기화**”로 연결

즉, 이 법은 “관광진흥” 자체뿐 아니라 **제재 체제 속에서 관광지구 개발이 어떤 방식으로 법적 평가/제한을 받는지**를 검토해야 하는 대상으로 제시됩니다(본문에 ‘제재 영향권’ 논리로 서술).

---


### 테스트 시나리오 2: 내부 문서에 없어서 웹 검색이 필요한 경우

In [7]:
query = '현재 2026년도 대한민국의 환율은 얼마이고, 최근 환율 정책에 대한 동향은 어떤가요?'
inputs = {'question':query}
config = {'recursion_limit':50}
result = app_vision.invoke(inputs, config)
print("==================================== 최종 답변 ===============================")
print(result['generation'])
print("=============================================================================")

[node:retrieve] 유사 요약 페이지 검색 시작...
[Node: retrieve] 검색 완료 매핑된 이미지 페이지수: 2
[node:grade_documents] 페이지 요약 적합성 판정...
[페이지 5] 심사 결과 : 부적합
[페이지 8] 심사 결과 : 부적합
 검색문서가 부적합함으로 실시간 인터넷검색을 가동합니다.
[Edge: Decide] 경로 분기 판단...
관련 비전 데이터 부재로 실시간 검색으로 우회....
[node: web_search] 실시간 포털 검색 가동중...
naver api 뉴스 검색중....
  - 네이버 뉴스 검색 결과 없음, DuckDuckGo로 전환
DuckDuckGo 백업 검색 가동중...
[node: generation] 비전 분석기 기반 답변 구성 중...
멀티모달 최종 답변 구성 완료
==================================== 최종 답변 ===============================
이미지(표/수치)를 확인할 수 없어, 요청하신 내용은 제공된 **텍스트 요약 컨텍스트**를 기준으로 답변드릴게요.

## 1) 2026년도 대한민국(원/달러) 환율 전망(현재로부터의 위치 포함)
- 2026년 원-달러 환율은 **대체로 1,350원~1,500원 범위**에서 움직일 가능성이 크다는 전망이 많습니다.  
- 특히 **1,400원대가 새로운 “뉴노멀(기준점)”로 자리 잡을 여지**가 있다는 의견이 우세합니다.
- 또 다른 관점에서는 현재 환율이 **약 1,440원대**에서 움직이고 있으며, **연말까지 1,450원 수준으로 쉽게 내려가기 어렵다**는 전망도 함께 제시됩니다.
- 종합적으로는 **1,300원대 후반 ~ 1,400원대 초반(상단 구간 포함)**에서 등락할 가능성이 높다는 시각이 있습니다.

## 2) 최근 환율 정책/동향(방향성)
텍스트 컨텍스트 기준으로 정리하면, 최근 동향은 다음 키워드로 요약됩니다.

- **환율은 단일 변수로 예측하기 어렵다 → 복수 시나리오 대응 필